In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

import plotly.express as px
import plotly.io as pio
pio.renderers.default = "browser"  # reliable

In [2]:
df = pd.read_csv("../data/naca4series_24_airfoils.csv")

print("Shape:", df.shape)
print("Reynolds values:", sorted(df["reynoldsNumber"].unique()))
print("Airfoils:", df["airfoilName"].nunique())

Shape: (16500, 8)
Reynolds values: [np.int64(50000), np.int64(100000), np.int64(200000), np.int64(500000), np.int64(1000000)]
Airfoils: 24


In [3]:
df = df.copy()
df["L_over_D"] = df["liftCoefficient"] / df["dragCoefficient"].replace(0, np.nan)

In [4]:
re_values = sorted(df["reynoldsNumber"].unique())

fig = go.Figure()

# Add traces for each Reynolds
for Re in re_values:
    d = df[df["reynoldsNumber"] == Re]
    for airfoil in d["airfoilName"].unique():
        d_air = d[d["airfoilName"] == airfoil]

        fig.add_trace(
            go.Scatter(
                x=d_air["angleOfAttack"],
                y=d_air["liftCoefficient"],
                mode="lines",
                name=f"{airfoil} (Re={Re})",
                visible=(Re == re_values[0])  # Only first Re visible initially
            )
        )

# Create dropdown buttons
buttons = []
n_airfoils = df["airfoilName"].nunique()

for i, Re in enumerate(re_values):
    visibility = [False] * (len(re_values) * n_airfoils)
    start = i * n_airfoils
    end = start + n_airfoils
    for j in range(start, end):
        visibility[j] = True

    buttons.append(
        dict(
            label=f"Re = {Re}",
            method="update",
            args=[{"visible": visibility},
                  {"title": f"Lift Curve at Re = {Re}"}]
        )
    )

fig.update_layout(
    updatemenus=[
        dict(
            active=0,
            buttons=buttons
        )
    ],
    title=f"Lift Curve at Re = {re_values[0]}",
    xaxis_title="Angle of Attack",
    yaxis_title="Lift Coefficient"
)

fig.show()